# Agentic Reels Studio — Transferencia de Baile 2D Anime GRATIS en Google Colab (GPU T4)

Notebook 1-clic de **Josué D. (`SadiTLM`)** para transferir bailes reales de tu carpeta `Muestras` a tu personaje 2D anime (*Bandida Bella*) usando la **GPU NVIDIA T4 gratuita de Google Colab ($0 API)**.

### Cómo usarlo en 4 pasos:
1. En Google Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.
2. Corre las celdas en orden (1 → 5).
3. Sube o selecciona desde Google Drive (`Muestras`) tu imagen **`bandida_greenscreen_ref.png`** y el clip de baile recortado (ej. `ref_dance_6s.mp4`).
4. Descarga o guarda en Drive el archivo final **`bandida_dance_alpha.webm`** (VP9 transparente) y súbelo al estudio Remotion (`chibi.video`) para añadir autos nocturnos, humo, glitch, `BANG!` y exportación dual con/sin audio.

In [ ]:
# 1. Verificar GPU T4 gratuita y montar Google Drive (opcional)
!nvidia-smi
import os, torch
assert torch.cuda.is_available(), 'Activa GPU T4 en Entorno de ejecución -> Cambiar tipo de entorno!'
print('GPU lista:', torch.cuda.get_device_name(0))

MOUNT_DRIVE = True  # Cambia a False si prefieres subir archivos manualmente
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# 2. Instalar MimicMotion (open-source pose-driven video transfer, Apache-2.0) + rembg/FFmpeg
!git clone --depth 1 https://github.com/Tencent/MimicMotion.git /content/MimicMotion
%cd /content/MimicMotion
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers==0.27.2 'huggingface_hub<0.26' omegaconf transformers accelerate einops decord av onnxruntime-gpu mediapipe opencv-python-headless rembg
# Parche para PyTorch >= 2.4 en Colab (corrige safe_globals(*allowed_modules) en Tencent/MimicMotion)
!python -c "p='mimicmotion/utils/loader.py'; s=open(p).read(); s=s.replace('with torch.serialization.safe_globals(*allowed_modules):', 'if True:'); open(p,'w').write(s)"
!mkdir -p models/DWPose
!wget -q -O models/DWPose/yolox_l.onnx https://huggingface.co/yzd-v/DWPose/resolve/main/yolox_l.onnx
!wget -q -O models/DWPose/dw-ll_ucoco_384.onnx https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.onnx
!wget -q -O models/MimicMotion_1-1.pth https://huggingface.co/tencent/MimicMotion/resolve/main/MimicMotion_1-1.pth
print('Modelos MimicMotion 1.1 descargados!')

In [ ]:
# 3. Elegir imagen 2D de personaje + video de baile de referencia
# Puedes subirlos directo o apuntar a tu carpeta de Google Drive
from pathlib import Path

REF_IMAGE = '/content/bandida_greenscreen_ref.png'
REF_VIDEO = '/content/ref_dance_6s.mp4'

if not Path(REF_IMAGE).exists() or not Path(REF_VIDEO).exists():
    from google.colab import files
    print('Sube bandida_greenscreen_ref.png y ref_dance_6s.mp4 (o edita las rutas arriba hacia /content/drive/MyDrive/Muestras/...):')
    uploaded = files.upload()
    for fname in uploaded:
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            REF_IMAGE = f'/content/MimicMotion/{fname}'
        elif fname.lower().endswith(('.mp4', '.mov')):
            REF_VIDEO = f'/content/MimicMotion/{fname}'

print('Imagen personaje:', REF_IMAGE)
print('Video baile:', REF_VIDEO)

In [ ]:
# 4. Ejecutar MimicMotion en GPU T4 (optimizado en fp16 para 15 GB VRAM)
import yaml

cfg = {
    'base_model_path': 'vdo/stable-video-diffusion-img2vid-xt-1-1',
    'ckpt_path': 'models/MimicMotion_1-1.pth',
    'test_case': [
        {
            'ref_video_path': REF_VIDEO,
            'ref_image_path': REF_IMAGE,
            'num_frames': 72,       # ~4.8s a 15fps (subir a 96 si sobra VRAM)
            'resolution': 576,
            'frames_overlap': 6,
            'num_inference_steps': 25,
            'noise_aug_strength': 0.0,
            'guidance_scale': 2.0,
            'sample_stride': 2,
            'fps': 15,
            'seed': 42,
        }
    ],
}
with open('configs/bandida_run.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

!python inference.py --inference_config configs/bandida_run.yaml --output_dir /content/outputs
!ls -lh /content/outputs

In [ ]:
# 5. Convertir el baile generado a .webm VP9 con transparencia alfa (yuva420p) listo para Remotion
import glob, subprocess, shutil
from pathlib import Path

mp4s = sorted(glob.glob('/content/outputs/**/*.mp4', recursive=True))
assert mp4s, 'No se encontro MP4 en /content/outputs'
raw_mp4 = mp4s[-1]
alpha_webm = '/content/bandida_dance_alpha.webm'

# Recorte chroma-key + despill gratis con FFmpeg (si el fondo salio verde #00FF00)
cmd = [
    'ffmpeg', '-y', '-i', raw_mp4,
    '-vf', 'colorkey=0x00FF00:0.28:0.10,despill=type=green:mix=0.55:expand=0.1,fps=30,format=yuva420p',
    '-c:v', 'libvpx-vp9', '-pix_fmt', 'yuva420p', '-auto-alt-ref', '0', '-b:v', '0', '-crf', '24', '-an',
    alpha_webm,
]
subprocess.run(cmd, check=True)
print('Listo para Remotion (chibi.video):', alpha_webm)

# Guardar copia en Drive si esta montado y descargar
drive_dest = Path('/content/drive/MyDrive/Muestras')
if drive_dest.exists():
    shutil.copy2(alpha_webm, drive_dest / 'bandida_dance_alpha.webm')
    print('Copiado automaticamente a Google Drive -> Muestras/bandida_dance_alpha.webm')

from google.colab import files
files.download(alpha_webm)